Este notebook analiza un conjunto de datos de incidentes delictivos reportados en la Ciudad de Chicago durante el año 2026 (hasta el dia 20 de Enero 2020), el cual incluye información sobre el tipo de delito, la fecha del incidente y si se realizo un arresto, junto con coordenadas de latitud y longitud que permiten su visualización geográfica.

El objetivo es utilizar Plotly Express para crear un mapa interactivo y explorar patrones geográficos del crimen dentro de la ciudad a partir de los datos disponibles.

**Variables del dataset Chicago Crimes (2026)**
* unique_key (number): Identificador único del registro dentro del conjunto de datos.
* case_number (text): Número RD del Departamento de Policía de Chicago, único para cada incidente reportado.
* **date (date/time):** Fecha y hora en que ocurrió el incidente. En algunos casos corresponde a una estimación.
* block (text): Dirección parcialmente anonimizada del incidente, presentada a nivel de bloque para proteger la privacidad de las víctimas.
* iucr (text): Código del Illinois Uniform Crime Reporting (IUCR), directamente relacionado con el tipo y la descripción del delito.
* **primary_type (text):** Descripción principal del delito según el código IUCR.
* description (text): Descripción secundaria del delito, correspondiente a una subcategoría del tipo principal.
* **location_description (text):** Descripción del tipo de lugar donde ocurrió el incidente.
* **arrest (boolean):** Indica si el incidente derivó en un arresto (true/false).
* domestic (boolean): Indica si el incidente está relacionado con violencia doméstica según el Illinois Domestic Violence Act.
* beat (text): Beat policial donde ocurrió el incidente. Es la unidad geográfica policial más pequeña con patrullaje asignado.
* district (text): Distrito policial en el que ocurrió el incidente.
* ward (number): Distrito del Concejo Municipal (ward) correspondiente a la ubicación del incidente.
* fbi_code (text): Clasificación del delito según el sistema National Incident-Based Reporting System (NIBRS) del FBI.
* x_coordinate (number): Coordenada X de la ubicación del incidente en la proyección State Plane Illinois East NAD 1983. La ubicación está desplazada por motivos de privacidad, pero se mantiene dentro del mismo bloque.
* y_coordinate (number): Coordenada Y de la ubicación del incidente en la proyección State Plane Illinois East NAD 1983, también desplazada para preservar la privacidad.
* year (number): Año en que ocurrió el incidente.
* updated_on (date/time): Fecha y hora de la última actualización del registro.
* **latitude (number):** Latitud de la ubicación del incidente. La coordenada está desplazada para proteger la privacidad, manteniéndose dentro del mismo bloque.
* **longitude (number):** Longitud de la ubicación del incidente. Al igual que la latitud, se encuentra desplazada por motivos de anonimización.
* location (location): Ubicación del incidente en un formato espacial que permite la creación de mapas y otros análisis geográficos.


# Chicago Crimes 2026 — Análisis Geolocalizado de Incidentes Delictivos

## Introducción

Este análisis explora los incidentes delictivos reportados en la Ciudad de 
Chicago durante el año 2026. A través de visualizaciones geolocalizadas 
e interactivas se busca responder las siguientes preguntas:

- ¿En qué zonas de Chicago se concentran los delitos?
- ¿Cuáles son los 5 tipos de delito más frecuentes y cómo se distribuyen?
- ¿Existen diferencias espaciales entre incidentes con y sin arresto?
- ¿Los delitos nocturnos se concentran en zonas distintas a los diurnos?

**Herramientas utilizadas:** Python · Pandas · Plotly  
**Dataset:** [Chicago Crimes 2026 — Data.gov](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

## 2. Carga del dataset

Se utiliza el archivo `Crimes_Chicago_2026.csv` con incidentes delictivos 
reportados en Chicago durante el año 2026 (hasta el 20 de enero).
A continuación se visualizan las primeras filas para entender su estructura.

In [ ]:
# Se carga el dataset
df = pd.read_csv("Crimes_Chicago_2026.csv")

# Se realiza una vista rápida para verificar los datos
df.head()

## 3. Limpieza de datos

Se realizan los siguientes pasos:
- Eliminación de filas sin coordenadas de latitud y/o longitud (0.36% del total)
- Conversión de `Latitude` y `Longitude` de tipo objeto a numérico
- Conversión de la variable `Date` a formato datetime

In [ ]:
#Se realiza un conteo de valores nulos por variable en porcentajes
(df.isnull().mean() * 100).round(2)

In [ ]:
#Dado que el porcentaje de valores faltantes es mínimo ( aprox 0.36%), las filas sin latitud y/o longitud no aportan a
#la visualización, por lo que se eliminarán. Esto no afecta significativamente al tamaño ni a la representatividad del dataset.

df_clean = df.dropna(subset=["Latitude", "Longitude"]).copy()


In [ ]:
#Se revisa que se hayan eliminado los duplicados
df_clean.duplicated().sum()

In [ ]:
#Tambien se observa que los registros de Latitute y Longitude son del tipo objeto, con la coma como separador decimal. Por lo tanto se
# reemplazaa la coma por punto y se convierte a numero.
for col in ["Latitude", "Longitude"]:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [ ]:
#Se eliminan las filas que hayan quedado invalidas por el paso anterior
df_clean = df_clean.dropna(subset=["Latitude", "Longitude"]).copy()

In [ ]:
#Se Verifica que haya quedado  todo bien
df_clean[["Latitude", "Longitude"]].head()
df_clean[["Latitude", "Longitude"]].dtypes

In [ ]:
# Ahora se convierten los registros de la variable "Date" a formato datetime
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

In [ ]:
# Se chequea que la variable "Date" efectivamente tenga el formato datetime
df_clean["Date"].dtype

In [ ]:
df_clean["Date"].head()

Para la creación del mapa interactivo geolocalizado con Plotly Express, se seleccionan las variables:

**Variables geográficas**
Estas variables permiten posicionar cada incidente en el mapa:
* latitude: latitud del lugar donde ocurrió el incidente.
* longitude: longitud del lugar donde ocurrió el incidente.

**Variables categóricas para análisis**: Estas variables se utilizan para agregar información contextual e interactividad al mapa.
* primary_type: permite diferenciar los incidentes según el tipo de delito mediante colores.
* location_description: describe el tipo de lugar donde ocurrió el incidente y se muestra en la información emergente (hover).
* arrest: indica si el incidente derivó en un arresto, lo que permite analizar diferencias espaciales entre incidentes con y sin arresto.

**Variables temporales (opcional, para análisis exploratorio)**: Estas variables permiten realizar filtros o análisis adicionale.
* date: posibilita filtrar los incidentes por período o analizar la distribución temporal de los delitos.


## 4. Visualizaciones

### 4.1 Mapa interactivo — Todos los tipos de delito

Se visualizan todos los incidentes del dataset en un mapa interactivo, 
diferenciando cada tipo de delito por color.

In [ ]:
# Mapa interactivo
fig = px.scatter_mapbox(
    df_clean,
    lat="Latitude",
    lon="Longitude",
    color="Primary Type",
    hover_name="Primary Type",
    hover_data={
        "Description": True,
        "Location Description": True,
        "Arrest": True,
        "Domestic": True,
        "Date": True,
        "Latitude": False,
        "Longitude": False
    },
    zoom=10,
    mapbox_style="carto-positron",
    title="Chicago Crimes (2026) — Mapa interactivo por tipo de delito"
)

fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig.show()
fig.write_html("mapa_todos_los_crimenes.html")

A partir del mapa interactivo se observa que los incidentes delictivos no se distribuyen de manera uniforme en la Ciudad de Chicago, sino que se concentran en determinadas zonas, formando clusters o hotspots de criminalidad. Estas áreas presentan una alta densidad de registros y corresponden principalmente a zonas de mayor actividad urbana.

Al diferenciar los incidentes por tipo de delito, se aprecia una superposición de múltiples categorías en las zonas más densas, lo que indica que distintos tipos de delitos coexisten en los mismos sectores. Por el contrario, en áreas más periféricas la cantidad de incidentes es menor y la diversidad de tipos de delitos también se reduce.

En conjunto, la visualización permite identificar patrones espaciales claros y demuestra la utilidad de los datos geolocalizados y de las visualizaciones interactivas para el análisis exploratorio del crimen urbano.

Se observa que no se logra apreciar bien debido a la cantidad de crimenes. Se decide filtrar el dataset con los 5 tipo de crimenes mas repetidos en la ciudad y crear un nuevo mapa

### 4.2 Mapa interactivo — Top 5 tipos de delito

Dado que la cantidad de incidentes dificulta la lectura del mapa anterior, 
se filtran únicamente los **5 tipos de delito más frecuentes** para una 
visualización más clara.

In [ ]:
# Se identifican los 5 delitos mas frecuentes
top5_crimes = (
    df_clean["Primary Type"]
    .value_counts()
    .head(5)
    .index
)

top5_crimes

In [ ]:
# Se crea un dataset secundario con los registros de estos delitos
df_top5 = df_clean[df_clean["Primary Type"].isin(top5_crimes)].copy()
df_top5.shape

In [ ]:
#Se crea un mapa interactivo con los 5 delitos mas frecuentes
fig = px.scatter_mapbox(
    df_top5,
    lat="Latitude",
    lon="Longitude",
    color="Primary Type",
    hover_name="Primary Type",
    hover_data=[
        "Description",
        "Location Description",
        "Arrest",
        "Domestic",
        "Date"
    ],
    zoom=10,
    mapbox_style="carto-positron",
    title="Chicago Crimes (2026) — Mapa interactivo (Top 5 tipos de delito)"
)

# Ajustes visuales recomendados
fig.update_traces(marker=dict(size=6, opacity=0.6))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))

fig.show()
fig.write_html("mapa_top5_crimenes.html")

A partir del mapa interactivo limitado a los cinco tipos de delito más frecuentes, se observa que los incidentes se concentran en determinadas zonas de la Ciudad de Chicago, formando hotspots de criminalidad. Estas concentraciones se presentan principalmente en áreas urbanas de mayor densidad y actividad.

La visualización evidencia una superposición espacial de distintos tipos de delitos en las zonas más afectadas, lo que sugiere que los delitos más comunes tienden a coexistir en los mismos sectores geográficos. En contraste, las áreas periféricas muestran una menor cantidad de incidentes y una distribución más dispersa.

En particular, delitos como Theft y Motor Vehicle Theft presentan una fuerte presencia en zonas específicas, mientras que Battery y Assault se distribuyen de manera más homogénea, aunque manteniendo concentraciones claras en los principales núcleos urbanos.

Una variante adicional, seria separar los incidentes segun si hubo arresto o no usando la variable 'arrest' como color del mapa.

### 4.3 Mapa interactivo — Incidentes con y sin arresto

Se analiza la distribución espacial de los incidentes diferenciando 
si derivaron o no en un arresto, utilizando la variable `Arrest` como color.

In [ ]:
#Se crea un nuevo dataset llamado df_arrest
df_arrest = df_clean.copy()
df_arrest.shape


In [ ]:
fig = px.scatter_mapbox(
    df_arrest,
    lat="Latitude",
    lon="Longitude",
    color="Arrest",
    hover_name="Primary Type",
    hover_data=[
        "Primary Type",
        "Description",
        "Location Description",
        "Domestic",
        "Date"
    ],
    zoom=10,
    mapbox_style="carto-positron",
    title="Chicago Crimes (2026) — Incidentes con y sin arresto"
)

# Ajustes visuales recomendados
fig.update_traces(marker=dict(size=6, opacity=0.6))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))

fig.show()
fig.write_html("mapa_arrestos.html")


Al analizar el mapa interactivo coloreado según la variable arrest, se observa que los incidentes con y sin arresto no se distribuyen de manera uniforme en el espacio urbano. Existen zonas con una alta concentración de delitos donde predominan los incidentes sin arresto, así como áreas específicas donde la proporción de incidentes con arresto es mayor.

En las zonas de mayor densidad delictiva se aprecia una superposición de ambos tipos de eventos, aunque los incidentes sin arresto tienden a ser más frecuentes. Por otro lado, los arrestos aparecen con mayor concentración en determinadas áreas puntuales, lo que podría estar relacionado con factores como la presencia policial, el tipo de delito predominante o las características del entorno urbano.

En conjunto, esta visualización permite identificar diferencias espaciales en la ocurrencia de arrestos y ejemplifica cómo las visualizaciones geolocalizadas e interactivas facilitan el análisis exploratorio de datos delictivos.

Otra variante seria determinar con el color de cada punto, si el delito fue realizado en la noche o durante el dia. definiendo:
Día: entre 06:00 y 18:00
Noche: entre 18:00 y 06:00

### 4.4 Mapa interactivo — Incidentes diurnos vs nocturnos

Se crea la variable `time_of_day` para clasificar cada incidente según 
el momento del día en que ocurrió:

- **Día:** entre las 06:00 y las 18:00
- **Noche:** entre las 18:00 y las 06:00

In [ ]:
# Se extrae hora
df_time = df_clean.copy()
df_time["hour"] = df_time["Date"].dt.hour

# Se crea la variable Día / Noche
df_time["time_of_day"] = df_time["hour"].apply(
    lambda x: "Night" if (x < 6 or x >= 18) else "Day"
)

df_time["time_of_day"].value_counts()


In [ ]:
# Mapa interactivo: color por Día / Noche

In [ ]:
fig = px.scatter_mapbox(
    df_time,
    lat="Latitude",
    lon="Longitude",
    color="time_of_day",   # <-- clave
    hover_name="Primary Type",
    hover_data=[
        "Primary Type",
        "Description",
        "Location Description",
        "Arrest",
        "Domestic",
        "Date"
    ],
    zoom=10,
    mapbox_style="carto-positron",
    title="Chicago Crimes (2026) — Incidentes diurnos vs nocturnos"
)

# Ajustes visuales
fig.update_traces(marker=dict(size=6, opacity=0.6))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))

fig.show()
fig.write_html("mapa_dia_noche.html")


> **Hallazgo:** Los delitos nocturnos se concentran con mayor intensidad 
> en determinados sectores urbanos, mientras que los incidentes diurnos 
> muestran una distribución más homogénea en toda la ciudad.

## 5. Conclusiones

En este trabajo se analizaron los incidentes delictivos reportados en la Ciudad de Chicago durante el año 2026 mediante visualizaciones geolocalizadas e interactivas desarrolladas con Plotly Express. El uso de mapas interactivos permitió identificar patrones espaciales relevantes y comparar la distribución de los delitos según si derivaron o no en arresto, así como según el momento del día en que ocurrieron.

El análisis evidenció que los incidentes sin arresto son más frecuentes y se concentran en diversas zonas de la ciudad, especialmente en áreas de alta densidad urbana, mientras que los delitos con arresto presentan una distribución más localizada, con concentraciones en zonas específicas. Esto sugiere que la ocurrencia de arrestos no es homogénea en el territorio y puede estar influenciada por factores operativos, contextuales o por el tipo de delito predominante.

Asimismo, al diferenciar los eventos diurnos y nocturnos, se observó una distribución relativamente equilibrada en términos de cantidad, aunque con patrones espaciales distintos. Los delitos nocturnos tienden a concentrarse con mayor intensidad en determinados sectores urbanos, mientras que los incidentes diurnos muestran una distribución más homogénea. La superposición de ambos tipos de eventos en las zonas más afectadas indica la presencia de actividad delictiva sostenida a lo largo del día.

En conjunto, estas visualizaciones demuestran el valor de las herramientas de visualización interactiva de datos geolocalizados para el análisis exploratorio del crimen urbano, facilitando la identificación de patrones espaciales y temporales y contribuyendo a una mejor comprensión de la dinámica delictiva en la ciudad.

---
*Análisis realizado por Juan Bautista Acuña — [GitHub](https://github.com/bautistaacuna)*